# End-to-End Docker Hallucination Pipeline (Fully Automated)

**Dataset → New Scenario Derivation → Multi-LLM Collection → Validation → Report**

This notebook runs the entire remaining workflow with no manual classification step:

- **Part 1** — downloads the Henkel et al. Dataset of Dockerfiles, parses `FROM` instructions,
  frequency-counts base images, categorizes them, and derives a list of NEW scenarios (technology
  categories not already covered by your original 22) using a fixed, reproducible threshold rule.
- **Part 2** — sends a fixed prompt for each new scenario to 5 LLMs via their APIs (ChatGPT, Claude,
  Gemini, DeepSeek, GPT-OSS-120B via Groq) and parses out the recommended `FROM image:tag`.
- **Part 3** — validates every recommendation against live Docker Hub + endoflife.date data and applies
  your 6-category priority classification rule (same logic already verified on your original 110 rows).
- **Part 4** — merges with your original results (optional), aggregates, and produces the final tables
  and charts.

**Two things you MUST check/edit before trusting the output — flagged clearly at each spot below:**

1. **`ORIGINAL_PROMPT_TEMPLATE`** in Part 2 — reconstructed from context, not copied from your actual
   original prompt wording. Edit it to match EXACTLY what you used for the original 22 scenarios, or
   the new batch won't be methodologically consistent with the old one.
2. **`ALREADY_COVERED_IMAGES`** in Part 1 — my best reconstruction of which base images your original
   22 scenarios already used, so they're excluded from "new" candidates. Please verify this list against
   your own scenario table.

Nothing here requires you to manually classify, verify, or eyeball any Docker Hub/EOL result — that part
is fully automated end to end, same as your existing validated pipeline.


> In this repository: the final combined results produced by this notebook, after the re-check in notebook 02, are `data/combined/combined_full_results_rechecked.csv`.


## Setup

In [ ]:
!pip install -q google-genai anthropic openai --quiet

import requests
import pandas as pd
import time
import re
import os
import json
import tarfile
import zipfile
import io
from collections import Counter
from datetime import datetime, date

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

TODAY = date.today()
print("Running as of:", TODAY)
print("PIPELINE: full end-to-end (dataset -> new scenarios -> multi-LLM -> validation)")


# PART 1 — Dataset Download, Frequency Analysis & New Scenario Derivation


### 1.1 Load your real frequency data

You already parsed the full 178,506-file Henkel dataset once (see `parsing_report.txt`:
178,202 files with a FROM instruction, 51,207 distinct image references, 22 categories). We reuse
that exact output instead of re-downloading/re-parsing ~178k files from scratch - faster, and
guarantees the numbers in this notebook match what you already reported.

Upload `base_image_frequency.csv` (51,207 rows: `image_ref, count`) below. If you don't have it handy,
this cell also accepts a fresh raw-Dockerfiles archive as a fallback (see the commented Path B code).


In [ ]:
from google.colab import files

print("Upload 'base_image_frequency.csv' (columns: image_ref, count)")
uploaded = files.upload()
freq_filename = next((k for k in uploaded if 'base_image_frequency' in k.lower()), list(uploaded.keys())[0])

base_freq_df = pd.read_csv(freq_filename)
print(f"Loaded {len(base_freq_df)} distinct image:tag references, sum of counts = {base_freq_df['count'].sum()}")
base_freq_df.head()


### 1.2 Aggregate to base-image-name granularity (strip tags)

In [ ]:
FROM_RE = re.compile(r'^\s*`{0,3}\s*FROM\s+(?:--platform=\S+\s+)?(\S+)', re.IGNORECASE | re.MULTILINE)

def get_base_name(image_ref):
    """Strip the tag/digest off an image reference, e.g. 'node:18-alpine' -> 'node'."""
    ref = str(image_ref).strip()
    if '@' in ref:
        ref = ref.split('@')[0]
    if ':' in ref.split('/')[-1]:
        ref = ref.rsplit(':', 1)[0]
    return ref

base_freq_df['base_name'] = base_freq_df['image_ref'].apply(get_base_name)
name_counts = base_freq_df.groupby('base_name')['count'].sum().sort_values(ascending=False)

print(f"Total distinct base image names: {len(name_counts)}")
print(f"Sum of counts (should match parsing_report.txt 'files with FROM'): {name_counts.sum()}")
print()
print("Top 20:")
name_counts.head(20)


### 1.3 Categorize the 'Other/Uncategorized' bucket and find NEW candidate scenarios

`ALREADY_COVERED_PREFIXES` = base images already used by your original 22 scenarios, plus the four
"Base OS" categories (Ubuntu/Alpine/Debian/CentOS/Fedora/Amazon Linux/Oracle Linux/openSUSE) and
`scratch`/`busybox`, which were never turned into application scenarios (a raw OS image isn't an
"application deployment scenario"). **Verify this set matches your own scenario table.**

`CATEGORY_KEYWORDS` covers 43 categories total (30 from the first pass + 13 more found by mining
deeper into the remaining 92,022 still-uncategorized occurrences: Windows Containers, Kubernetes
Networking, Service Mesh/Istio, GPU/CUDA, Tomcat, Maven, JBoss/WildFly, Clojure, Fluentd, Config
Management, Conda/Data Science, IoT/Embedded, Helm).

Matching uses **word-boundary regex**, not plain substring search - plain substring matching produced
false positives (e.g. 'vipconsult/base' wrongly matching 'consul', 'jtreminio/php' wrongly matching
'minio'); word-boundary matching was tested against the real data and confirmed to eliminate these.


In [ ]:
ALREADY_COVERED_PREFIXES = {
    "node", "python", "golang", "go", "eclipse-temurin", "openjdk", "java", "php", "ruby", "rust",
    "postgres", "mysql", "redis", "mongo", "rabbitmq", "nginx",
    "tensorflow/tensorflow", "pytorch/pytorch", "jupyter/scipy-notebook", "jupyter/datascience-notebook",
    "mcr.microsoft.com/dotnet/aspnet", "docker.elastic.co/elasticsearch/elasticsearch",
    "ubuntu", "alpine", "debian", "centos", "scratch", "busybox",
    "fedora", "amazonlinux", "oraclelinux", "opensuse", "registry.access.redhat.com",
    "microsoft/dotnet", "mcr.microsoft.com/dotnet", "dotnet/aspnet", "dotnet/sdk", "dotnet/runtime",
    "elasticsearch",
}

def is_covered(name):
    n = name.lower()
    return any(n == p or n.startswith(p + "/") or n.startswith(p) for p in ALREADY_COVERED_PREFIXES)

# 43 categories total - verified against your real base_image_frequency.csv
CATEGORY_KEYWORDS = {
    "R Language": ["r-base", "rocker"], "Jenkins/CI": ["jenkins"], "Elixir/Erlang": ["elixir", "erlang"],
    "Etcd": ["etcd"], "Grafana/Monitoring": ["grafana", "prometheus", "influxdb", "telegraf"],
    "Scala/Kotlin (JVM)": ["scala", "kotlin", "gradle"], "MariaDB": ["mariadb"],
    "Apache HTTPD": ["httpd", "apache2"], "C/C++ Build": ["gcc", r"g\+\+", "clang", "cmake"],
    "Perl": ["perl"], "WordPress/CMS": ["wordpress", "drupal", "joomla", "prestashop"],
    "HAProxy": ["haproxy"], "Selenium/Testing": ["selenium", "playwright", "cypress"],
    "Kafka": ["kafka", "zookeeper"], "Haskell": ["haskell", "ghc"], "Swift": ["swift"],
    "Vault/Consul": ["vault", "consul"], "Caddy/Traefik (Proxy)": ["caddy", "traefik"],
    "Cassandra": ["cassandra"], "Solr/Search": ["solr"], "Nextcloud": ["nextcloud"],
    "Julia": ["julia"], "MinIO/Object Storage": ["minio"], "Memcached": ["memcached"],
    "CouchDB": ["couchdb"], "Swarm/Orchestration": ["swarm", "portainer"], "ActiveMQ": ["activemq"],
    "Deno": ["denoland/deno", "deno"], "Nats": ["nats"], "SQLite": ["sqlite"],
    "Windows Containers": ["microsoft/windowsservercore", "microsoft/nanoserver", "microsoft/aspnet",
                            "microsoft/vsts-agent", "mcr.microsoft.com/windows"],
    "Kubernetes Networking (CNI)": ["quay.io/calico", "quay.io/coreos/flannel"],
    "Service Mesh (Istio)": ["gcr.io/istio-release"],
    "GPU/CUDA Base": ["nvidia/cuda"],
    "Apache Tomcat": ["tomcat"],
    "Maven Build": ["maven"],
    "JBoss/WildFly": ["jboss/wildfly", "jboss"],
    "Clojure": ["clojure"],
    "Fluentd/Logging": ["fluent/fluentd", "fluentd"],
    "Config Mgmt (Ansible/Puppet)": ["williamyeh/ansible", "devopsil/puppet", "ansible", "puppet"],
    "Conda/Data Science": ["continuumio/miniconda", "floydhub"],
    "IoT/Embedded (Balena)": ["balenalib"],
    "Helm/K8s Package Mgr": ["kubernetes-helm/tiller", "alpine/helm"],
}

def _make_pattern(kw):
    return re.compile(r'(?<![a-z0-9])' + kw + r'(?![a-z0-9])', re.IGNORECASE)

_COMPILED = {cat: [_make_pattern(kw) for kw in kws] for cat, kws in CATEGORY_KEYWORDS.items()}

def categorize(name):
    n = name.lower()
    for cat, patterns in _COMPILED.items():
        if any(p.search(n) for p in patterns):
            return cat
    return None

rows = []
for name, cnt in name_counts.items():
    if is_covered(name):
        continue
    cat = categorize(name)
    if cat:
        rows.append({"base_name": name, "count": cnt, "category": cat})

candidates_df = pd.DataFrame(rows).sort_values("count", ascending=False)

CATEGORY_FREQUENCY_THRESHOLD = 20  # same reproducible-threshold style as your original scenario selection
category_totals = candidates_df.groupby("category")["count"].sum().sort_values(ascending=False)
qualifying_categories = category_totals[category_totals >= CATEGORY_FREQUENCY_THRESHOLD].index.tolist()

print(f"{len(qualifying_categories)} of {len(category_totals)} candidate categories meet the "
      f"threshold (total occurrences >= {CATEGORY_FREQUENCY_THRESHOLD})")
category_totals[category_totals >= CATEGORY_FREQUENCY_THRESHOLD]


### 1.4 Split qualifying categories into up to 2 scenarios each (grounded in real distinct images)

Same approach your original 22 scenarios already used for Node.js (REST API + WebSocket), Java (Spring
Boot + batch), PHP (web app + Laravel) - a technology category can legitimately map to more than one
scenario when the data shows two genuinely distinct, high-frequency images within it. The 2nd scenario
per category is only added if its image individually clears `MIN_SECONDARY_COUNT` occurrences, so this
never manufactures a scenario from noise. `NOISE_PREFIXES` filters out CPU-architecture variants and
CI-vendor forks first, so representatives are images a developer would actually reach for.

**This threshold/splitting logic was fixed BEFORE looking at any validation results** - scenario
selection stays outcome-blind, driven only by real Henkel-dataset frequency, never by which scenarios
might produce more or fewer hallucinations. That independence is what makes the eventual error-rate
numbers trustworthy.


In [ ]:
NOISE_PREFIXES = ("ppc64le/", "s390x/", "arm64v8/", "arm32v7/", "i386/", "amd64/",
                   "circleci/", "drone/", "travisci/")

def is_noise(name):
    n = name.lower()
    return any(n.startswith(p) for p in NOISE_PREFIXES)

MAX_SCENARIOS_PER_CATEGORY = 2
MIN_SECONDARY_COUNT = 15

new_scenarios_rows = []
for cat in category_totals.loc[qualifying_categories].sort_values(ascending=False).index:
    cat_rows = candidates_df[(candidates_df["category"] == cat) & (~candidates_df["base_name"].apply(is_noise))]
    if cat_rows.empty:
        cat_rows = candidates_df[candidates_df["category"] == cat]
    cat_rows = cat_rows.sort_values("count", ascending=False)

    picked, seen_cores = [], set()
    for _, r in cat_rows.iterrows():
        core = r["base_name"].split("/")[-1]  # crude de-dup key (avoids near-identical variants)
        if core in seen_cores:
            continue
        seen_cores.add(core)
        picked.append(r)
        if len(picked) >= MAX_SCENARIOS_PER_CATEGORY:
            break

    for i, r in enumerate(picked):
        if i > 0 and r["count"] < MIN_SECONDARY_COUNT:
            continue
        new_scenarios_rows.append({
            "scenario_name": f"{cat} application using {r['base_name']}",
            "category": cat,
            "image_name": r["base_name"],
            "frequency_count": int(r["count"]),
            "category_total_occurrences": int(category_totals[cat]),
        })

new_scenarios = pd.DataFrame(new_scenarios_rows).sort_values("category_total_occurrences", ascending=False).reset_index(drop=True)

# --- Data-driven mainstream/niche tag (decided by frequency alone, fixed before seeing any results) ---
MAINSTREAM_THRESHOLD = 500  # category_total_occurrences >= this -> 'Mainstream', else 'Niche'
new_scenarios["popularity"] = new_scenarios["category_total_occurrences"].apply(
    lambda x: "Mainstream" if x >= MAINSTREAM_THRESHOLD else "Niche")

new_scenarios.to_csv("new_scenarios_derived.csv", index=False)
print(f"Derived {len(new_scenarios)} new scenarios -> saved to new_scenarios_derived.csv")
print(f"GRAND TOTAL scenarios (22 original + {len(new_scenarios)} new) = {22 + len(new_scenarios)}")
print()
print(new_scenarios['popularity'].value_counts())
new_scenarios


# PART 2 — Automated Multi-LLM Data Collection

Sends a fixed prompt for each new scenario to all 5 LLMs via their APIs.

**Edit `ORIGINAL_PROMPT_TEMPLATE` below to match your actual original wording exactly.**


### 2.1 API keys

Uses Colab's Secrets manager (key icon in the left sidebar) so keys are never typed into a cell.
Add secrets named exactly: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`, `DEEPSEEK_API_KEY`,
`XAI_API_KEY` (or `GROQ_API_KEY` if using the free Groq/Llama option below). Any LLM whose key is missing is automatically skipped (not silently zero-filled).


In [ ]:
from google.colab import userdata

def get_secret(name):
    try:
        val = userdata.get(name)
        if not val:
            print(f"  [{name}] userdata.get() returned empty/None - secret may not exist under this exact name")
        return val
    except Exception as e:
        print(f"  [{name}] FAILED: {type(e).__name__}: {e}")
        return None

print("Attempting to read each secret (errors will be shown, not hidden):")
API_KEYS = {
    "ChatGPT":  get_secret("OPENAI_API_KEY"),
    "Claude":   get_secret("ANTHROPIC_API_KEY"),
    "Gemini":   get_secret("GOOGLE_API_KEY"),
    "DeepSeek": get_secret("DEEPSEEK_API_KEY"),
    "GPT-OSS-120B": get_secret("GROQ_API_KEY"),  # free via Groq (was "Grok" via paid xAI XAI_API_KEY)
}

ENABLED_LLMS = [llm for llm, key in API_KEYS.items() if key]
print()
print("Enabled LLMs (API key found):", ENABLED_LLMS)
print("Skipped (no API key in Colab secrets):", [llm for llm in API_KEYS if llm not in ENABLED_LLMS])
print()
print("If you see a small 'Grant access' notification/popup appear near the top of Colab after ")
print("running this cell, click it for EACH secret, then re-run this cell.")


In [ ]:
# Your exact original prompt template (as given), with {scenario} substituted per scenario:
ORIGINAL_PROMPT_TEMPLATE = (
    "Write a Dockerfile FROM instruction for a {scenario}. Only provide the FROM line."
)


In [ ]:
import openai
import anthropic
from google import genai as google_genai  # new SDK (old google.generativeai was EOL Aug 2025)

if API_KEYS["ChatGPT"]:
    openai_client = openai.OpenAI(api_key=API_KEYS["ChatGPT"])
if API_KEYS["Claude"]:
    anthropic_client = anthropic.Anthropic(api_key=API_KEYS["Claude"])
if API_KEYS["Gemini"]:
    gemini_client = google_genai.Client(api_key=API_KEYS["Gemini"])

# DeepSeek and GPT-OSS-120B (via Groq) both expose OpenAI-compatible chat/completions endpoints
if API_KEYS["DeepSeek"]:
    deepseek_client = openai.OpenAI(api_key=API_KEYS["DeepSeek"], base_url="https://api.deepseek.com")
if API_KEYS["GPT-OSS-120B"]:
    groq_client = openai.OpenAI(api_key=API_KEYS["GPT-OSS-120B"], base_url="https://api.groq.com/openai/v1")


def call_chatgpt(prompt):
    resp = openai_client.chat.completions.create(
        model="gpt-5.6",  # current recommended API model as of Aug 2026 (was gpt-5.1, outdated)
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content


def call_claude(prompt):
    resp = anthropic_client.messages.create(
        model="claude-sonnet-5",  # correct current API model string (confirmed exact, no date suffix)
        max_tokens=1024,  # bumped from 300 - Sonnet 5 has adaptive thinking on by default, which
                           # needs headroom in the token budget even for a short final answer
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in resp.content if hasattr(b, "text"))


def call_gemini(prompt):
    resp = gemini_client.models.generate_content(
        model="gemini-3.6-flash",  # per Google's own error message: 2.5-flash closed to new users, use 3.6-flash
        contents=prompt,
    )
    return resp.text


def call_deepseek(prompt):
    resp = deepseek_client.chat.completions.create(
        model="deepseek-v4-flash",  # was "deepseek-chat", retired July 24, 2026 - this is the successor
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content


def call_gpt_oss(prompt):
    resp = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",  # free tier on Groq (confirmed active; llama-3.3-70b-versatile was deprecated Jun 17, 2026)
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content


CALL_FUNCS = {
    "ChatGPT": call_chatgpt,
    "Claude": call_claude,
    "Gemini": call_gemini,
    "DeepSeek": call_deepseek,
    "GPT-OSS-120B": call_gpt_oss,
}


### 2.2 Extract `FROM image:tag` from a raw LLM response

In [ ]:
def parse_from_line(response_text):
    """Pull the first FROM instruction's image:tag out of raw LLM text (handles code fences)."""
    m = FROM_RE.search(response_text)
    if not m:
        return None, None
    ref = m.group(1).strip().strip('`')
    if ':' in ref.split('/')[-1]:
        image, tag = ref.rsplit(':', 1)
    else:
        image, tag = ref, "latest"
    return image, tag


### 2.2b Smoke test - 1 call per LLM before the full run

Run this BEFORE the full 275-call loop below. It costs only 5 API calls total and immediately shows
whether each provider/model/billing combination actually works, so a bad model string or an inactive
billing account fails fast instead of burning through the whole batch.


In [ ]:
_test_prompt = ORIGINAL_PROMPT_TEMPLATE.format(scenario="Node.js REST API service")
print("Smoke test - 1 call per enabled LLM:\n")
for llm in ENABLED_LLMS:
    try:
        resp = CALL_FUNCS[llm](_test_prompt)
        img, tag = parse_from_line(resp)
        status = "OK" if img else "OK (but no FROM line parsed - check Raw Response)"
        print(f"[{status}] {llm}: {resp.strip()[:80]}")
    except Exception as e:
        print(f"[FAILED] {llm}: {e}")
print("\nFix any [FAILED] lines above (check model string / billing) before running the full loop below.")


### 2.3 Run collection over all new scenarios x enabled LLMs

In [ ]:
from tqdm.notebook import tqdm

collection_results = []
scenario_llm_pairs = [(s, llm) for s in new_scenarios.to_dict("records") for llm in ENABLED_LLMS]

for scenario_row, llm in tqdm(scenario_llm_pairs):
    scenario_name = scenario_row["scenario_name"]
    prompt = ORIGINAL_PROMPT_TEMPLATE.format(scenario=scenario_name)
    meta = {"category": scenario_row.get("category"), "popularity": scenario_row.get("popularity")}

    try:
        raw_response = CALL_FUNCS[llm](prompt)
    except Exception as e:
        collection_results.append({
            "LLM": llm, "Scenario": scenario_name, "Image Name": None, "Tag": None,
            "Raw Response": f"ERROR: {e}", **meta,
        })
        continue

    image, tag = parse_from_line(raw_response)
    collection_results.append({
        "LLM": llm, "Scenario": scenario_name, "Image Name": image, "Tag": tag,
        "Raw Response": raw_response, **meta,
    })
    time.sleep(1)  # basic rate-limit courtesy across providers

collection_df = pd.DataFrame(collection_results)
collection_df.insert(0, "#", range(1, len(collection_df) + 1))
collection_df.to_csv("new_scenarios_collection.csv", index=False)
print(f"Collected {len(collection_df)} new LLM responses.")
collection_df.head(15)


### 2.4 Retry only the failed rows

If the summary above shows failures (API errors, rate limits, or a row where `Image Name` is blank),
you do NOT need to re-run the full batch. This cell finds every failed row and retries only those,
then patches the results back into `collection_df`.

**Note on quota errors specifically:** some providers' free tiers cap requests *per day*, not just
per minute (e.g. Gemini's free tier for `gemini-3.6-flash` allows only 20 requests/day at the time of
writing - check the exact number in your own error message, as quotas change). If a quota is genuinely
exhausted for the day, retrying immediately won't help - come back after the quota resets (usually at
midnight in the provider's timezone) and re-run this cell then. To retry in a **fresh Colab session**
on a later day, run Parts 1-2.1 as normal, then upload your saved `new_scenarios_collection.csv` in the
next cell instead of re-collecting everything from scratch.


In [ ]:
# Optional: if you're in a NEW session and want to resume from a previously saved collection file,
# upload it here. If collection_df already exists in this session (you just ran 2.3), skip this cell.
try:
    collection_df
    print(f"Using existing collection_df from this session ({len(collection_df)} rows).")
except NameError:
    print("No collection_df in memory - upload your saved new_scenarios_collection.csv:")
    resume_upload = files.upload()
    resume_csv = next((k for k in resume_upload if k.lower().endswith('.csv')), None)
    collection_df = pd.read_csv(resume_csv)
    print(f"Loaded {len(collection_df)} rows from {resume_csv}.")


In [ ]:
failed_mask = collection_df["Image Name"].isna()
n_failed = failed_mask.sum()
print(f"{n_failed} of {len(collection_df)} rows currently failed.")
if n_failed > 0:
    print(collection_df[failed_mask]["LLM"].value_counts())


In [ ]:
from tqdm.notebook import tqdm as tqdm_retry

retry_rows = collection_df[failed_mask].copy()
retry_count = 0
skipped_count = 0

for idx, row in tqdm_retry(retry_rows.iterrows(), total=len(retry_rows)):
    llm = row["LLM"]
    if llm not in ENABLED_LLMS:
        skipped_count += 1
        continue  # this LLM's key isn't loaded in this session - can't retry it here

    scenario_name = row["Scenario"]
    prompt = ORIGINAL_PROMPT_TEMPLATE.format(scenario=scenario_name)

    try:
        raw_response = CALL_FUNCS[llm](prompt)
        image, tag = parse_from_line(raw_response)
        collection_df.loc[idx, "Image Name"] = image
        collection_df.loc[idx, "Tag"] = tag
        collection_df.loc[idx, "Raw Response"] = raw_response
        if image is not None:
            retry_count += 1
    except Exception as e:
        collection_df.loc[idx, "Raw Response"] = f"ERROR: {e}"
    time.sleep(1)

collection_df.to_csv("new_scenarios_collection.csv", index=False)
still_failed = collection_df["Image Name"].isna().sum()
print(f"Retried {len(retry_rows)} rows -> {retry_count} newly succeeded, {still_failed} still failing"
      f" ({skipped_count} skipped because that LLM has no key loaded this session).")
print("Re-run this '2.4' section again (or come back later for quota resets) until still_failed == 0,")
print("or accept the remainder as-is and proceed - Part 3 will simply skip any row with no Image Name.")


# PART 3 — Automated Validation (identical logic to your already-verified pipeline)

Same Docker Hub + endoflife.date checks and 6-category priority rule already validated on your
original 110 rows (81.8% agreement with manual classification, with remaining differences traced to
manual staleness rather than pipeline errors) - plus the new automated rolling-tag digest resolution
so this batch needs zero manual review steps.


In [ ]:
DOCKERHUB_API = "https://hub.docker.com/v2"
_dockerhub_cache = {}

def parse_image_ref(image_name):
    parts = image_name.strip().split('/')
    if len(parts) == 1:
        return None, None, parts[0]
    first = parts[0]
    looks_like_host = ('.' in first) or (':' in first) or (first == 'localhost')
    if looks_like_host:
        registry = first
        namespace = parts[1] if len(parts) > 2 else None
        repo = parts[-1]
        return registry, namespace, repo
    else:
        namespace = parts[0]
        repo = parts[-1]
        return None, namespace, repo


def _http_get(url):
    for attempt in range(4):
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 429:
                time.sleep(3 * (attempt + 1))
                continue
            return r
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
    return None


def dockerhub_check(image_name, tag):
    cache_key = (image_name, tag)
    if cache_key in _dockerhub_cache:
        return _dockerhub_cache[cache_key]

    registry_host, namespace, repo = parse_image_ref(image_name)
    result = {"registry_host": registry_host, "namespace": namespace, "repo": repo,
              "image_exists": None, "tag_exists": None, "is_official": None}

    if registry_host is not None:
        result["is_official"] = False
        _dockerhub_cache[cache_key] = result
        return result

    ns = namespace if namespace else "library"
    result["is_official"] = (namespace is None)

    repo_url = f"{DOCKERHUB_API}/namespaces/{ns}/repositories/{repo}/"
    r = _http_get(repo_url)
    result["image_exists"] = True if (r and r.status_code == 200) else (False if (r and r.status_code == 404) else None)

    if result["image_exists"]:
        tag_url = f"{DOCKERHUB_API}/namespaces/{ns}/repositories/{repo}/tags/{tag}/"
        r2 = _http_get(tag_url)
        result["tag_exists"] = True if (r2 and r2.status_code == 200) else (False if (r2 and r2.status_code == 404) else None)
    else:
        result["tag_exists"] = False

    time.sleep(0.3)
    _dockerhub_cache[cache_key] = result
    return result


def get_tag_digest(image_name, tag):
    """Fetch a tag's content digest, used to resolve fully-unversioned rolling tags (see 3.x below)."""
    registry_host, namespace, repo = parse_image_ref(image_name)
    if registry_host is not None:
        return None
    ns = namespace if namespace else "library"
    r = _http_get(f"{DOCKERHUB_API}/namespaces/{ns}/repositories/{repo}/tags/{tag}/")
    if r and r.status_code == 200:
        try:
            return r.json().get("digest")
        except Exception:
            return None
    return None


def list_all_tags(image_name, max_pages=5):
    """List up to max_pages*100 tags for an image, newest-updated first."""
    registry_host, namespace, repo = parse_image_ref(image_name)
    if registry_host is not None:
        return []
    ns = namespace if namespace else "library"
    tags = []
    url = f"{DOCKERHUB_API}/namespaces/{ns}/repositories/{repo}/tags/?page_size=100&ordering=-last_updated"
    for _ in range(max_pages):
        r = _http_get(url)
        if not r or r.status_code != 200:
            break
        data = r.json()
        tags.extend(data.get("results", []))
        url = data.get("next")
        if not url:
            break
    return tags


### 3.1 endoflife.date functions (with automated rolling-tag resolution)

For fully unversioned tags (e.g. `nginx:alpine`, `redis:alpine` - no digits at all), instead of flagging
for manual review, this resolves the tag automatically: it fetches the tag's digest and compares it
against the digests of all versioned tags for the same image, finding which specific version it
currently matches. This closes the last manual-review gap from your validated pipeline, at the cost of
a few extra API calls per unversioned tag.


In [ ]:
EOL_PRODUCT_MAP = {
    "node": "nodejs", "python": "python", "golang": "go", "go": "go", "php": "php", "ruby": "ruby",
    "rust": "rust", "postgres": "postgresql", "mysql": "mysql", "redis": "redis", "mongo": "mongodb",
    "rabbitmq": "rabbitmq", "nginx": "nginx", "eclipse-temurin": "eclipse-temurin",
    "mariadb": "mariadb", "httpd": "apache", "wordpress": "wordpress", "drupal": "drupal",
    "elixir": "elixir", "erlang": "erlang", "haskell": "haskell", "grafana/grafana": "grafana",
    "prom/prometheus": "prometheus", "jenkins/jenkins": "jenkins", "jenkins": "jenkins",
    "cassandra": "cassandra", "solr": "solr", "haproxy": "haproxy", "r-base": "r",
}

_eol_cache = {}

def fetch_eol_cycles(product_slug):
    if product_slug in _eol_cache:
        return _eol_cache[product_slug]
    try:
        r = requests.get(f"https://endoflife.date/api/{product_slug}.json", timeout=15)
        data = r.json() if r.status_code == 200 else None
    except requests.RequestException:
        data = None
    _eol_cache[product_slug] = data
    return data


def extract_version(tag):
    m = re.match(r'^(\d+(?:\.\d+)*)', tag.strip())
    return m.group(1) if m else None


def _cycle_sort_key(entry):
    parts = str(entry.get('cycle', '')).split('.')
    key = []
    for p in parts:
        try:
            key.append(int(p))
        except ValueError:
            key.append(0)
    return tuple(key)


def resolve_rolling_tag(image_name, tag):
    """For a fully unversioned tag (no digits), find which versioned tag shares its digest.
    Returns a resolved version string, or None if it can't be determined."""
    target_digest = get_tag_digest(image_name, tag)
    if not target_digest:
        return None
    all_tags = list_all_tags(image_name, max_pages=3)
    for t in all_tags:
        name = t.get("name", "")
        if name == tag:
            continue
        if re.match(r'^\d', name) and t.get("digest") == target_digest:
            return extract_version(name)
    return None


def check_eol_status(image_name, tag):
    if tag.strip().lower() == 'latest':
        return "UNKNOWN", "tag is 'latest' - handled separately as Risky Latest Tag"

    version = extract_version(tag)
    resolved_note = ""
    if version is None:
        resolved = resolve_rolling_tag(image_name, tag)
        if resolved:
            version = resolved
            resolved_note = f" (unversioned tag '{tag}' resolved via digest match to version {resolved})"
        else:
            return "UNKNOWN", f"tag '{tag}' has no version and digest-match resolution failed - needs manual check"

    slug = EOL_PRODUCT_MAP.get(image_name)
    if not slug:
        return "UNKNOWN", f"no endoflife.date mapping configured for image '{image_name}' - needs manual check"

    cycles = fetch_eol_cycles(slug)
    if not cycles:
        return "UNKNOWN", f"could not fetch endoflife.date data for product '{slug}'"

    version_parts = version.split('.')
    candidates = ['.'.join(version_parts[:i]) for i in range(len(version_parts), 0, -1)]
    cycle_match = None
    for cand in candidates:
        exact = [e for e in cycles if str(e.get('cycle', '')) == cand]
        if exact:
            cycle_match = exact[0]
            break

    matched_by_prefix = False
    if not cycle_match:
        prefix_matches = [e for e in cycles if str(e.get('cycle', '')).startswith(version + '.')]
        if prefix_matches:
            prefix_matches.sort(key=_cycle_sort_key, reverse=True)
            cycle_match = prefix_matches[0]
            matched_by_prefix = True

    if not cycle_match:
        return "UNKNOWN", f"version '{version}' not found in endoflife.date cycles for '{slug}'{resolved_note} - needs manual check"

    cyc_label = cycle_match.get('cycle')
    prefix_note = f" (rolling tag resolved to latest matching cycle {cyc_label})" if matched_by_prefix else ""
    eol_field = cycle_match.get('eol')

    if eol_field is False:
        return "SUPPORTED", f"cycle {cyc_label} - eol=false (still supported){prefix_note}{resolved_note}"
    if eol_field is True:
        return "EOL", f"cycle {cyc_label} - marked eol=true{prefix_note}{resolved_note}"
    if isinstance(eol_field, str):
        try:
            eol_date = datetime.strptime(eol_field, "%Y-%m-%d").date()
            status = "EOL" if eol_date < TODAY else "SUPPORTED"
            return status, f"cycle {cyc_label} - eol date {eol_field}{prefix_note}{resolved_note}"
        except ValueError:
            return "UNKNOWN", f"could not parse eol date '{eol_field}'"
    return "UNKNOWN", "unexpected eol field format"


### 3.2 Classification (your exact 6-priority rule)

In [ ]:
def classify_row(image_name, tag):
    image_name = str(image_name).strip()
    tag = str(tag).strip()
    dh = dockerhub_check(image_name, tag)

    if dh["registry_host"] is not None:
        return "OUT OF SCOPE", f"external registry: {dh['registry_host']}", dh, ("N/A", "external registry")

    if dh["image_exists"] is False:
        return "Non-Existent Image", "image not found on Docker Hub", dh, ("N/A", "image does not exist")
    if dh["image_exists"] is None:
        return "NEEDS MANUAL CHECK", "Docker Hub lookup failed for image", dh, ("N/A", "")

    if dh["tag_exists"] is False:
        return "Nonexistent Tag", "tag not found on Docker Hub", dh, ("N/A", "tag does not exist")
    if dh["tag_exists"] is None:
        return "NEEDS MANUAL CHECK", "Docker Hub lookup failed for tag", dh, ("N/A", "")

    if not dh["is_official"]:
        return "Non-Official Image", f"namespace '{dh['namespace']}' - not Docker Official Images", dh, ("N/A", "non-official")

    if tag.lower() == "latest":
        return "Risky Latest Tag", "tag is literally 'latest'", dh, ("N/A", "latest tag")

    eol_status, eol_note = check_eol_status(image_name, tag)
    if eol_status == "EOL":
        return "Outdated Tag", eol_note, dh, (eol_status, eol_note)
    elif eol_status == "SUPPORTED":
        return "Valid Official Image", eol_note, dh, (eol_status, eol_note)
    else:
        return "NEEDS MANUAL CHECK", eol_note, dh, (eol_status, eol_note)


### 3.3 Run validation over the new-scenario collection

In [ ]:
from tqdm.notebook import tqdm as tqdm2

validated_results = []
for _, row in tqdm2(collection_df.iterrows(), total=len(collection_df)):
    image_name, tag = row.get("Image Name"), row.get("Tag")
    if pd.isna(image_name) or str(image_name).strip() == "":
        continue
    auto_category, note, dh_info, eol_info = classify_row(image_name, tag)
    validated_results.append({
        "#": row.get("#"), "LLM": row.get("LLM"), "Scenario": row.get("Scenario"),
        "Scenario Category": row.get("category"), "Popularity": row.get("popularity"),
        "Image Name": image_name, "Tag": tag,
        "Automated Category": auto_category, "Automated Note": note,
        "Docker Registry": dh_info.get("registry_host") or "hub.docker.com",
        "Is Official (Docker Hub)": dh_info.get("is_official"),
        "EOL Status": eol_info[0], "EOL Note": eol_info[1],
    })

new_validated_df = pd.DataFrame(validated_results)
new_validated_df.to_csv("new_scenarios_validated.csv", index=False)
print(f"Validated {len(new_validated_df)} new rows.")
new_validated_df.head(15)


# PART 4 — Merge with Original Results, Aggregate, and Report

Optionally combines the new-scenario batch with your original 22-scenario/110-row results for one
unified final report covering all scenarios.


In [ ]:
print("Upload your ORIGINAL validated results file (automated_results_full.csv) to merge, "
      "or skip this cell to report on the new-scenario batch only.")
orig_uploaded = files.upload()


In [ ]:
orig_csv = next((k for k in orig_uploaded if k.lower().endswith('.csv')), None) if orig_uploaded else None

# Popularity tag for the ORIGINAL 22 scenarios, derived the same data-driven way (category total
# occurrences from your real application_category_frequency.csv, threshold >= 500 = Mainstream),
# so all 240 combined rows use one consistent rule.
ORIGINAL_IMAGE_POPULARITY = {
    "node": "Mainstream", "python": "Mainstream", "golang": "Mainstream", "eclipse-temurin": "Mainstream",
    "openjdk": "Mainstream", "php": "Mainstream", "ruby": "Mainstream", "rust": "Mainstream",
    "postgres": "Mainstream", "mysql": "Mainstream", "nginx": "Mainstream",
    "tensorflow/tensorflow": "Mainstream", "pytorch/pytorch": "Mainstream",
    "jupyter/scipy-notebook": "Mainstream", "jupyter/datascience-notebook": "Mainstream",
    "mcr.microsoft.com/dotnet/aspnet": "Mainstream",
    "redis": "Niche", "mongo": "Niche", "rabbitmq": "Niche",
    "docker.elastic.co/elasticsearch/elasticsearch": "Niche",
}

if orig_csv:
    orig_df = pd.read_csv(orig_csv)
    orig_df["Popularity"] = orig_df["Image Name"].map(ORIGINAL_IMAGE_POPULARITY).fillna("Mainstream")
    orig_df["Batch"] = "Original (22 scenarios, manual web-UI collection)"
    new_validated_df["Batch"] = "Expansion (55 scenarios, automated API collection)"
    keep_cols = ["LLM", "Scenario", "Image Name", "Tag", "Automated Category", "Automated Note", "Popularity", "Batch"]
    combined_df = pd.concat([orig_df[keep_cols], new_validated_df[keep_cols]], ignore_index=True)
    print(f"Merged: {len(orig_df)} original rows + {len(new_validated_df)} new rows = {len(combined_df)} total")
else:
    combined_df = new_validated_df.copy()
    combined_df["Batch"] = "Expansion (55 scenarios, automated API collection)"
    print(f"No original file uploaded - reporting on {len(combined_df)} new rows only")

combined_df.to_csv("combined_final_results.csv", index=False)


## Results Dashboard

Everything below is generated fresh from `combined_df` - re-running these cells after re-running
Parts 1-3 always regenerates matching numbers, tables, and charts with no manual copy-pasting.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

CATEGORY_COLORS = {
    "Valid Official Image": "#16a34a",
    "Outdated Tag": "#f59e0b",
    "Non-Official Image": "#3b82f6",
    "Risky Latest Tag": "#f97316",
    "Nonexistent Tag": "#dc2626",
    "Non-Existent Image": "#991b1b",
    "NEEDS MANUAL CHECK": "#6b7280",
}


In [ ]:
in_scope_df = combined_df[combined_df["Automated Category"] != "OUT OF SCOPE"].copy()

agg = in_scope_df["Automated Category"].value_counts().reset_index()
agg.columns = ["Category", "Count"]
agg["%"] = (agg["Count"] / len(in_scope_df) * 100).round(1)
agg = agg.sort_values("Count", ascending=False)
print(f"In-scope rows: {len(in_scope_df)}  (out-of-scope rows excluded: {(combined_df['Automated Category']=='OUT OF SCOPE').sum()})")
agg


In [ ]:
per_llm = pd.crosstab(in_scope_df["LLM"], in_scope_df["Automated Category"])
per_scenario_group = in_scope_df.groupby("Scenario")["Automated Category"].value_counts().unstack(fill_value=0)
per_llm


### Figure 1 - Overall classification breakdown (all in-scope responses)

In [ ]:
fig, ax = plt.subplots(figsize=(9, max(4.5, len(agg) * 0.55)))
agg_sorted = agg.sort_values("Count", ascending=True)
bar_colors = [CATEGORY_COLORS.get(c, "#94a3b8") for c in agg_sorted["Category"]]
bars = ax.barh(agg_sorted["Category"], agg_sorted["Count"], color=bar_colors)
for bar, cnt, pct in zip(bars, agg_sorted["Count"], agg_sorted["%"]):
    ax.text(bar.get_width() + max(agg["Count"]) * 0.01, bar.get_y() + bar.get_height() / 2,
            f"{cnt} ({pct}%)", va="center", fontsize=10)
ax.set_xlabel("Number of responses")
ax.set_title(f"Docker Image Recommendation Classification\n(n={len(in_scope_df)} in-scope responses across all scenarios and LLMs)",
             fontsize=12, fontweight="bold")
ax.margins(x=0.15)
plt.tight_layout()
plt.savefig("fig1_overall_classification.png", bbox_inches="tight")
plt.show()


### Figure 2 - Per-LLM comparison (% of each LLM's own responses)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
category_order = ([c for c in CATEGORY_COLORS if c in per_llm.columns] +
                   [c for c in per_llm.columns if c not in CATEGORY_COLORS])
per_llm_pct = per_llm[category_order].div(per_llm.sum(axis=1), axis=0) * 100
colors = [CATEGORY_COLORS.get(c, "#94a3b8") for c in category_order]
per_llm_pct.plot(kind="bar", stacked=True, ax=ax, color=colors, width=0.7)
ax.set_ylabel("% of that LLM's in-scope responses")
ax.set_xlabel("")
ax.set_title("Per-LLM Classification Breakdown", fontsize=12, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9, title="Category")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig("fig2_per_llm_breakdown.png", bbox_inches="tight")
plt.show()


### Figure 3 - Mainstream vs. Niche technology comparison

Tests the hypothesis that LLMs hallucinate more on less-common technology stacks. `Popularity` was
assigned purely from real Henkel-dataset frequency (>= 500 total occurrences = Mainstream), fixed
during scenario derivation in Part 1 - **before** any of these results existed, so this comparison is
a genuine discovered finding, not a scenario selection designed to produce it either way.


In [ ]:
ERROR_CATEGORIES = {"Outdated Tag", "Non-Official Image", "Risky Latest Tag",
                     "Nonexistent Tag", "Non-Existent Image", "NEEDS MANUAL CHECK"}

pop_summary = in_scope_df.groupby("Popularity").apply(
    lambda g: pd.Series({
        "total_rows": len(g),
        "valid_rate_%": round((g["Automated Category"] == "Valid Official Image").sum() / len(g) * 100, 1),
        "error_rate_%": round((g["Automated Category"].isin(ERROR_CATEGORIES).sum() / len(g)) * 100, 1),
    })
)
print("(error_rate_% = share of responses that were NOT 'Valid Official Image')")
pop_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, col, title, color in zip(
    axes, ["valid_rate_%", "error_rate_%"],
    ["Valid Official Image rate", "Hallucination/staleness rate"],
    ["#16a34a", "#dc2626"],
):
    bars = ax.bar(pop_summary.index, pop_summary[col], color=color, width=0.5)
    for bar, v in zip(bars, pop_summary[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 1, f"{v}%", ha="center", fontweight="bold")
    ax.set_ylabel("%")
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0, 100)
fig.suptitle("Mainstream vs. Niche Technologies", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig3_mainstream_vs_niche.png", bbox_inches="tight")
plt.show()


### Figure 4 - Original vs. Expansion batch comparison

Sanity check across your two collection methods (manual web-UI for the original 22 scenarios vs.
automated API for the 55 expansion scenarios) - useful for the Threats to Validity discussion of
whether collection method itself introduced any systematic difference.


In [ ]:
if combined_df["Batch"].nunique() > 1:
    batch_summary = in_scope_df.groupby("Batch")["Automated Category"].value_counts(normalize=True).mul(100).round(1).unstack(fill_value=0)
    batch_summary = batch_summary[[c for c in CATEGORY_COLORS if c in batch_summary.columns] +
                                   [c for c in batch_summary.columns if c not in CATEGORY_COLORS]]
    print(batch_summary.to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = [CATEGORY_COLORS.get(c, "#94a3b8") for c in batch_summary.columns]
    batch_summary.plot(kind="barh", stacked=True, ax=ax, color=colors)
    ax.set_xlabel("% of responses in that batch")
    ax.set_title("Original (Manual) vs. Expansion (Automated) Batch Comparison", fontsize=12, fontweight="bold")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.savefig("fig4_batch_comparison.png", bbox_inches="tight")
    plt.show()
else:
    print("Only one batch present (original results not merged in) - skipping batch comparison.")
    batch_summary = pd.DataFrame()


## Export everything

In [ ]:
with pd.ExcelWriter("combined_final_report.xlsx") as writer:
    combined_df.to_excel(writer, sheet_name="Combined Full Results", index=False)
    agg.to_excel(writer, sheet_name="Aggregate Summary", index=False)
    per_llm.to_excel(writer, sheet_name="Per-LLM Breakdown")
    pop_summary.to_excel(writer, sheet_name="Mainstream vs Niche")
    per_scenario_group.to_excel(writer, sheet_name="Per-Scenario Breakdown")
    if not batch_summary.empty:
        batch_summary.to_excel(writer, sheet_name="Original vs Expansion Batch")
    new_scenarios.to_excel(writer, sheet_name="Derived New Scenarios", index=False)
    category_totals.reset_index().to_excel(writer, sheet_name="Category Frequency Analysis", index=False)

print("Saved: combined_final_report.xlsx")
print("Saved: fig1_overall_classification.png, fig2_per_llm_breakdown.png,")
print("       fig3_mainstream_vs_niche.png" + (", fig4_batch_comparison.png" if not batch_summary.empty else ""))

from google.colab import files as colab_files
colab_files.download("combined_final_report.xlsx")
colab_files.download("fig1_overall_classification.png")
colab_files.download("fig2_per_llm_breakdown.png")
colab_files.download("fig3_mainstream_vs_niche.png")
if not batch_summary.empty:
    colab_files.download("fig4_batch_comparison.png")


> **Note:** this preliminary 12-row repair case study was in the original submission. It is superseded by the 79-attempt repair study in `03_robustness_and_repair_runs.ipynb` (Table IV of the paper); its old output is kept in `data/repair/preliminary_repair_case_study_n12.csv`.

# PART 5 — Preliminary Repair Case Study

A small, honest add-on beyond pure measurement: for a sample of flagged (risky/outdated/hallucinated)
recommendations, we re-prompt the *same* LLM that produced the original flagged answer, asking it to
suggest a safer replacement, then validate the new suggestion through the exact same pipeline
(`classify_row`) already used for the main dataset - no new validation logic, so results stay directly
comparable. This tests whether LLMs can self-correct their own risky recommendations, or whether
"repair" attempts introduce a *new* hallucination (a "self-repair hallucination" risk).

This is deliberately scoped small (a case study, not a full second RQ) - reuses everything already
built in Parts 2-3, so it only needs the cells below.


### 5.1 Select flagged rows to repair

Samples up to `N_REPAIR_SAMPLES` rows from the final combined results, stratified across the three
"repairable" categories (Outdated Tag, Risky Latest Tag, Nonexistent Tag) so the case study isn't
dominated by one failure type. Uses a fixed random seed for reproducibility.


In [ ]:
N_REPAIR_SAMPLES = 15
REPAIR_CATEGORIES = ["Outdated Tag", "Risky Latest Tag", "Nonexistent Tag"]
RANDOM_SEED = 42

repairable_pool = combined_df[combined_df["Automated Category"].isin(REPAIR_CATEGORIES)].copy()
print(f"{len(repairable_pool)} flagged rows available across {REPAIR_CATEGORIES}")
print(repairable_pool["Automated Category"].value_counts())

# Stratified sample: draw proportionally from each category, capped at N_REPAIR_SAMPLES total
per_cat_n = max(1, N_REPAIR_SAMPLES // len(REPAIR_CATEGORIES))
sampled_parts = []
for cat in REPAIR_CATEGORIES:
    subset = repairable_pool[repairable_pool["Automated Category"] == cat]
    n = min(per_cat_n, len(subset))
    if n > 0:
        sampled_parts.append(subset.sample(n=n, random_state=RANDOM_SEED))
repair_sample = pd.concat(sampled_parts).head(N_REPAIR_SAMPLES).reset_index(drop=True)

print(f"\nSelected {len(repair_sample)} rows for the repair case study:")
repair_sample[["LLM", "Scenario", "Image Name", "Tag", "Automated Category"]]


### 5.2 Re-prompt the same LLM for a safer replacement, then re-validate

Reuses `CALL_FUNCS`, `parse_from_line`, and `classify_row` from Parts 2-3 - no new logic. Only rows
whose original LLM still has an API key loaded in this session can be repaired (others are skipped
and reported, not silently dropped).


In [ ]:
REPAIR_PROMPT_TEMPLATE = (
    "This Dockerfile uses {image}:{tag}, which may be outdated, unpinned, or otherwise risky. "
    "Suggest a safer, currently-supported replacement. Only provide the FROM line."
)

repair_results = []
for _, row in repair_sample.iterrows():
    llm = row["LLM"]
    original_image, original_tag, original_category = row["Image Name"], row["Tag"], row["Automated Category"]

    if llm not in ENABLED_LLMS:
        repair_results.append({
            "LLM": llm, "Original Image:Tag": f"{original_image}:{original_tag}",
            "Original Category": original_category,
            "Repaired Image:Tag": None, "Repaired Category": "SKIPPED (no API key this session)",
            "Outcome": "Skipped",
        })
        continue

    prompt = REPAIR_PROMPT_TEMPLATE.format(image=original_image, tag=original_tag)
    try:
        raw_response = CALL_FUNCS[llm](prompt)
        new_image, new_tag = parse_from_line(raw_response)
    except Exception as e:
        repair_results.append({
            "LLM": llm, "Original Image:Tag": f"{original_image}:{original_tag}",
            "Original Category": original_category,
            "Repaired Image:Tag": None, "Repaired Category": f"ERROR: {e}",
            "Outcome": "API Error",
        })
        continue

    if new_image is None:
        repair_results.append({
            "LLM": llm, "Original Image:Tag": f"{original_image}:{original_tag}",
            "Original Category": original_category,
            "Repaired Image:Tag": None, "Repaired Category": "Could not parse FROM line",
            "Outcome": "Parse Failure",
        })
        continue

    new_category, new_note, _, _ = classify_row(new_image, new_tag)

    if new_category == "Valid Official Image":
        outcome = "Repaired successfully"
    elif new_category in REPAIR_CATEGORIES or new_category in {"Non-Existent Image", "Non-Official Image"}:
        outcome = "Still flagged / new hallucination introduced"
    else:
        outcome = "Uncertain (needs manual check)"

    repair_results.append({
        "LLM": llm, "Original Image:Tag": f"{original_image}:{original_tag}",
        "Original Category": original_category,
        "Repaired Image:Tag": f"{new_image}:{new_tag}", "Repaired Category": new_category,
        "Outcome": outcome,
    })
    time.sleep(1)

repair_df = pd.DataFrame(repair_results)
repair_df.to_csv("repair_case_study.csv", index=False)
repair_df


### 5.3 Summary

In [ ]:
attempted = repair_df[~repair_df["Outcome"].isin(["Skipped"])]
print(f"Attempted repairs: {len(attempted)} of {len(repair_df)} sampled rows "
      f"({len(repair_df) - len(attempted)} skipped - LLM not available this session)")
print()
print(attempted["Outcome"].value_counts())
print()

success_rate = (attempted["Outcome"] == "Repaired successfully").mean() * 100 if len(attempted) else 0
print(f"Repair success rate: {success_rate:.1f}%")
print()
print("Rows where repair introduced a NEW/different problem (self-repair hallucination candidates):")
attempted[attempted["Outcome"] == "Still flagged / new hallucination introduced"]


In [ ]:
from google.colab import files as colab_files_repair
colab_files_repair.download("repair_case_study.csv")
print("Saved and downloaded: repair_case_study.csv")
